# FedSignal — Multi-Country Central Bank NLP

**Countries covered:** United States (Fed), United Kingdom (BoE), Pakistan (SBP), Australia (RBA)

**Architecture:** One DistilBERT model fine-tuned per country on domain-expert-labeled sentences drawn from each central bank's own communication style. Each model predicts the 60-day change in the country's 1-year government bond yield from statement sentiment + the local short-rate-to-policy-rate spread.

### Structure
1. Shared imports & seeds  
2. Country configuration (data sources, scraping targets, statement labels)  
3. Training sentence banks — one per country  
4. Per-country pipeline: data → regime → target → NLP model → evaluation  
5. Cross-country comparison dashboard

## Part 1 — Shared Imports & Seeds

In [ ]:
import os
import pandas as pd
import numpy as np
import sys
import random
import warnings
warnings.filterwarnings('ignore')

import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from fredapi import Fred

# ── Import statement loader from Statements pull.py (same directory) ──────
from pathlib import Path
import importlib.util

_scraper_path = Path('../src/statements_pull.py')
_spec = importlib.util.spec_from_file_location('statements_pull', _scraper_path)
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

load_statements   = _mod.load_statements
scrape_statements = _mod.scrape_statements
get_labels        = _mod.get_labels

# Reproducibility
random.seed(24)
import numpy as np; np.random.seed(24)
torch.manual_seed(24)

FRED_KEY   = os.environ['FRED_API_KEY']  # set FRED_API_KEY in your environment
fred       = Fred(api_key=FRED_KEY)
device     = torch.device('cpu')
MODEL_NAME = 'distilbert-base-uncased'

print("Imports complete.")
print(f"Statements pull.py loaded from: {_scraper_path}")


## Part 2 — Country Configuration

Each entry defines how to collect statements, which FRED series to use for yields/rates, and what the local policy body is called.

In [ ]:
COUNTRY_CONFIG = {

    # ── UNITED STATES ──────────────────────────────────────────────────────
    'US': {
        'name': 'United States (Fed)',
        'body': 'FOMC',
        'currency': 'USD',
        'policy_rate_series': 'DFF',          # Effective Fed Funds Rate
        'one_year_series':    'DGS1',          # 1-year Treasury
        'two_year_series':    'DGS2',          # 2-year Treasury (for spread)
        'cpi_series':         'CPIAUCSL',
        'unrate_series':      'UNRATE',
        'gdp_series':         'A191RL1Q225SBEA',
        'scrape': 'fomc',                      # handled by dedicated scraper
        'inflation_target': 2.0,
    },

    # ── UNITED KINGDOM ─────────────────────────────────────────────────────
    'UK': {
        'name': 'United Kingdom (BoE)',
        'body': 'MPC',
        'currency': 'GBP',
        'policy_rate_series': 'BOERUKM',       # BoE official Bank Rate (monthly)
        'one_year_series':    'IRLTLT01GBM156N',# UK 1-year govt bond (monthly proxy)
        'two_year_series':    None,             # will use 1Y - Bank Rate spread
        'cpi_series':         'GBRCPIALLMINMEI',
        'unrate_series':      'LRHUTTTTGBM156S',
        'gdp_series':         'CLVMNACSCAB1GQGB',
        'scrape': 'boe',
        'inflation_target': 2.0,
    },

    # ── PAKISTAN ───────────────────────────────────────────────────────────
    'PK': {
        'name': 'Pakistan (SBP)',
        'body': 'MPC',
        'currency': 'PKR',
        'policy_rate_series': 'IRSTCB01PKM156N',   # SBP policy rate
        'one_year_series':    'IRLTLT01PKM156N',    # Pakistan 1-year T-bill proxy
        'two_year_series':    None,
        'cpi_series':         'PAKCP1020M086NEST',
        'unrate_series':      'LRUN64TTPKM156S',
        'gdp_series':         'PAKNGDPRPCPPPT',
        'scrape': 'sbp',
        'inflation_target': 5.0,                   # SBP medium-term target
    },

    # ── AUSTRALIA ──────────────────────────────────────────────────────────
    'AU': {
        'name': 'Australia (RBA)',
        'body': 'RBA Board',
        'currency': 'AUD',
        'policy_rate_series': 'RBXMNYCBD',         # RBA cash rate target
        'one_year_series':    'IR3TIB01AUM156N',   # Australia 3-month interbank (proxy)
        'two_year_series':    None,
        'cpi_series':         'AUSCPIALLMINMEI',
        'unrate_series':      'LRUN64TTAUM156S',
        'gdp_series':         'AUSGDPRQPSMEI',
        'scrape': 'rba',
        'inflation_target': 2.5,                   # RBA 2-3% band midpoint
    },
}

print("Country configs loaded:", list(COUNTRY_CONFIG.keys()))


## Part 3 — Training Sentence Banks

Each country gets its own set of hand-labeled sentences (0 = very dovish, 1 = very hawkish). The US bank is carried over from the original notebook. Pakistan, UK, and Australia are written to reflect each central bank's specific communication style, vocabulary, and institutional framing.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  US — Federal Open Market Committee (FOMC)
# ════════════════════════════════════════════════════════════════════════════
sentence_training_US = [
    # === RATE DECISIONS ===
    ("The Committee decided to raise the target range for the federal funds rate by 75 basis points", 0.95),
    ("The Committee decided to raise the target range for the federal funds rate by 50 basis points", 0.85),
    ("The Committee decided to raise the target range for the federal funds rate by 25 basis points", 0.75),
    ("The Committee decided to maintain the target range for the federal funds rate", 0.50),
    ("The Committee decided to lower the target range for the federal funds rate by 25 basis points", 0.25),
    ("The Committee decided to lower the target range for the federal funds rate by 50 basis points", 0.15),
    ("The Committee expects it will soon be appropriate to raise the target range for the federal funds rate", 0.85),
    ("It would be premature to begin lowering rates at this time", 0.80),

    # === FORWARD GUIDANCE - HAWKISH ===
    ("The Committee anticipates that ongoing increases in the target range will be appropriate", 0.90),
    ("The Committee is strongly committed to returning inflation to its 2 percent objective", 0.80),
    ("The Committee is prepared to raise rates further if appropriate", 0.85),
    ("Additional policy firming may be appropriate", 0.80),
    ("The Committee will continue reducing its holdings of Treasury securities and agency debt", 0.75),
    ("The Committee decided to continue to reduce the monthly pace of its net asset purchases", 0.70),
    ("Ongoing increases in the target range will be appropriate to bring inflation down", 0.85),
    ("The Committee will take into account the cumulative tightening of monetary policy", 0.65),

    # === FORWARD GUIDANCE - DOVISH ===
    ("The Committee is prepared to adjust the stance of monetary policy as appropriate if risks emerge", 0.35),
    ("The Committee would be prepared to adjust the stance of monetary policy as appropriate", 0.40),
    ("The Committee judges that the risks have become more balanced", 0.30),
    ("The risks to achieving its employment and inflation goals are roughly in balance", 0.30),
    ("The Committee is attentive to the risks to both sides of its dual mandate", 0.35),
    ("In considering additional adjustments the Committee will carefully assess incoming data", 0.40),
    ("The Committee does not expect it will be appropriate to reduce the target range until it has gained greater confidence", 0.55),

    # === INFLATION - HAWKISH ===
    ("Inflation remains elevated", 0.80),
    ("Inflation remains well above the Committee's 2 percent objective", 0.85),
    ("Inflation remains somewhat elevated", 0.65),
    ("Inflation continues to run well above the longer-run goal", 0.85),
    ("Core inflation remains elevated and persistent", 0.85),
    ("Price pressures remain broad based across sectors", 0.80),
    ("Inflation has shown little progress toward the 2 percent objective", 0.85),
    ("Supply and demand imbalances have continued to contribute to elevated levels of inflation", 0.75),
    ("Wage growth remains elevated relative to what would be consistent with price stability", 0.80),
    ("Housing costs continue to put upward pressure on overall inflation", 0.75),
    ("Inflation expectations risk becoming unanchored", 0.90),

    # === INFLATION - DOVISH ===
    ("Inflation has eased over the past year but remains above the Committee's target", 0.55),
    ("Inflation has made progress toward the Committee's 2 percent objective", 0.30),
    ("Inflation has made further progress toward the Committee's 2 percent objective", 0.25),
    ("Inflation has declined significantly from its peak", 0.20),
    ("The disinflationary process has continued", 0.20),
    ("Inflation has come down considerably over the past year", 0.25),
    ("Longer-term inflation expectations remain well anchored", 0.35),
    ("Inflation has moved closer to the Committee's 2 percent objective", 0.30),

    # === LABOR MARKET - HAWKISH ===
    ("The labor market remains extremely tight", 0.80),
    ("Job gains have been robust in recent months", 0.75),
    ("Job gains have been strong", 0.70),
    ("The unemployment rate has remained low", 0.65),
    ("Demand continues to outstrip supply in the labor market", 0.80),
    ("Consumer spending has been particularly strong", 0.75),
    ("Consumer spending has been surprisingly resilient", 0.70),
    ("Job gains have been solid in recent months and the unemployment rate has declined substantially", 0.70),

    # === LABOR MARKET - DOVISH ===
    ("Job gains have moderated", 0.35),
    ("Job gains have slowed", 0.30),
    ("The unemployment rate has moved up but remains low", 0.35),
    ("The labor market is showing signs of softening", 0.25),
    ("Job gains have remained low and the unemployment rate has shown some signs of stabilization", 0.35),
    ("Employment gains have slowed significantly", 0.20),

    # === ECONOMIC ACTIVITY - HAWKISH ===
    ("Economic activity has been expanding at a strong pace", 0.75),
    ("Economic activity has been expanding at a solid pace", 0.65),
    ("Available indicators suggest that economic activity has been expanding at a solid pace", 0.65),
    ("The economy has been stronger than expected", 0.75),
    ("Recent indicators suggest that economic activity has continued to expand at a solid pace", 0.65),
    ("Indicators of economic activity and employment have continued to strengthen", 0.70),

    # === ECONOMIC ACTIVITY - DOVISH ===
    ("Economic activity has slowed from its strong pace", 0.35),
    ("Economic activity has slowed considerably", 0.20),
    ("Growth has moderated from its earlier pace", 0.30),
    ("Recent indicators point to modest growth in spending and production", 0.35),
    ("The path of the economy continues to depend on the course of the virus", 0.30),
    ("Economic activity expanded at a moderate rate", 0.40),

    # === FINANCIAL CONDITIONS ===
    ("Overall financial conditions remain accommodative", 0.30),
    ("Financial conditions have tightened significantly", 0.25),
    ("Credit conditions have become more restrictive for households and businesses", 0.25),
    ("Tighter credit conditions for households and businesses are likely to weigh on economic activity", 0.20),
    ("The Federal Reserve's ongoing purchases will continue to foster smooth market functioning and accommodative financial conditions", 0.30),

    # === BALANCE SHEET ===
    ("The Committee will continue reducing its holdings of Treasury securities", 0.70),
    ("The Committee decided to begin reducing its holdings at a pace of 95 billion per month", 0.75),
    ("The Committee decided to slow the pace of decline of its securities holdings", 0.35),
    ("Beginning in February the Committee will increase its holdings of Treasury securities", 0.20),
    ("The Committee will increase its holdings of agency mortgage-backed securities", 0.20),

    # === RISKS ===
    ("Risks to the economic outlook remain including from new variants of the virus", 0.25),
    ("Uncertainty about the economic outlook has increased", 0.30),
    ("Uncertainty about the economic outlook remains elevated", 0.35),
    ("The Committee sees the risks to achieving its goals as roughly in balance", 0.40),
    ("Risks to the economic outlook are weighted to the downside", 0.15),
    ("Upside risks to inflation remain", 0.80),

    # === BOILERPLATE - NEUTRAL ===
    ("The Committee seeks to achieve maximum employment and inflation at the rate of 2 percent over the longer run", 0.50),
    ("The Committee will continue to monitor the implications of incoming information for the economic outlook", 0.50),
    ("The Committee's assessments will take into account a wide range of information", 0.50),
    ("In assessing the appropriate stance of monetary policy the Committee will continue to monitor", 0.50),
]

print(f"US training sentences: {len(sentence_training_US)}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  UK — Bank of England Monetary Policy Committee (MPC)
#  Communication style: formal, cautious, dual-mandate framing (CPI 2%)
#  Key phrases: "Bank Rate", "MPC", "gilt purchases", "Agent's Summary"
# ════════════════════════════════════════════════════════════════════════════
sentence_training_UK = [
    # === RATE DECISIONS ===
    ("The MPC voted to increase Bank Rate by 50 basis points to", 0.90),
    ("The MPC voted to increase Bank Rate by 25 basis points", 0.75),
    ("The MPC voted unanimously to increase Bank Rate by 25 basis points", 0.80),
    ("The MPC voted to maintain Bank Rate at its current level", 0.50),
    ("The MPC voted by a majority to hold Bank Rate unchanged", 0.50),
    ("The MPC voted to reduce Bank Rate by 25 basis points", 0.25),
    ("The MPC voted to reduce Bank Rate by 50 basis points", 0.15),
    ("The MPC voted to reduce Bank Rate to its effective lower bound", 0.05),
    ("Several members judged that a further tightening of monetary policy was warranted", 0.85),
    ("Some members preferred an immediate reduction in Bank Rate", 0.20),

    # === FORWARD GUIDANCE - HAWKISH ===
    ("The MPC will take the actions necessary to return inflation to the 2 percent target sustainably", 0.85),
    ("Further tightening in monetary policy may be required for a sustainable return of inflation to target", 0.85),
    ("The MPC stands ready to respond forcefully as necessary", 0.80),
    ("The Committee judges that monetary policy is likely to need to be restrictive for an extended period", 0.80),
    ("If there were to be evidence of more persistent inflationary pressures the MPC would respond forcefully", 0.85),
    ("The pace of future rate increases will depend on the evolution of the economic outlook", 0.65),

    # === FORWARD GUIDANCE - DOVISH ===
    ("The MPC will adjust monetary policy as warranted by economic data", 0.40),
    ("The Committee notes that monetary policy is not on a pre-set path", 0.40),
    ("Future decisions will be sensitive to the incoming economic data", 0.40),
    ("The MPC judges it appropriate to begin unwinding some of the monetary stimulus it has provided", 0.55),
    ("The MPC will keep the stance of monetary policy under review", 0.35),
    ("A gradual and limited approach to removing monetary policy stimulus remains appropriate", 0.35),

    # === INFLATION - HAWKISH ===
    ("CPI inflation remains well above the 2 percent target", 0.85),
    ("Inflation is expected to remain elevated for longer than previously anticipated", 0.80),
    ("Services inflation remains high and wage growth is above levels consistent with the inflation target", 0.85),
    ("Core goods inflation has remained elevated", 0.75),
    ("Second-round effects on domestic price and wage setting are evident", 0.80),
    ("The risks to inflation are judged to be skewed to the upside", 0.80),
    ("Inflation expectations have drifted above levels consistent with the target", 0.85),
    ("Domestic inflationary pressures have proved more persistent than expected", 0.80),

    # === INFLATION - DOVISH ===
    ("CPI inflation has fallen back towards the 2 percent target", 0.25),
    ("Inflation is expected to return sustainably to the 2 percent target", 0.25),
    ("There are increasing signs that the restrictive monetary policy stance is working", 0.30),
    ("Services price inflation has eased somewhat", 0.35),
    ("Headline CPI inflation is projected to decline materially over the coming months", 0.20),
    ("Inflation expectations have remained well anchored at longer horizons", 0.30),
    ("The contribution of energy prices to inflation has diminished", 0.25),

    # === LABOR MARKET - HAWKISH ===
    ("The labor market remains tight by historical standards", 0.75),
    ("Pay growth remains elevated", 0.75),
    ("Annual private sector regular pay growth has remained above rates consistent with the inflation target", 0.80),
    ("Underlying wage dynamics suggest inflationary wage-price dynamics remain a risk", 0.85),
    ("Vacancies remain elevated relative to unemployment", 0.70),

    # === LABOR MARKET - DOVISH ===
    ("The labor market has shown signs of easing", 0.30),
    ("The unemployment rate has risen slightly", 0.30),
    ("Indicators of pay growth have moderated somewhat although they remain elevated", 0.40),
    ("Job vacancies have fallen from their peak", 0.30),
    ("The labor market has loosened but remains relatively tight", 0.40),

    # === ECONOMIC ACTIVITY - HAWKISH ===
    ("UK GDP growth has been stronger than expected", 0.70),
    ("Consumer spending has been resilient despite elevated interest rates", 0.70),
    ("Output has continued to recover at a pace above pre-pandemic trends", 0.65),
    ("Demand conditions remain tight relative to supply capacity", 0.75),

    # === ECONOMIC ACTIVITY - DOVISH ===
    ("UK GDP is expected to grow only modestly", 0.30),
    ("GDP growth has been below potential and the output gap is slightly negative", 0.25),
    ("Business investment has been subdued in part reflecting Brexit-related uncertainty", 0.25),
    ("The economy has experienced a technical recession", 0.10),
    ("Economic activity is expected to remain weak over the near term", 0.20),
    ("Consumer confidence has fallen sharply", 0.20),

    # === FINANCIAL CONDITIONS / QE ===
    ("The MPC has voted to maintain the stock of UK government bond purchases at the current level", 0.50),
    ("The MPC has voted to reduce the stock of UK government bond purchases", 0.65),
    ("The MPC has voted to increase the stock of UK government bond purchases by 150 billion pounds", 0.15),
    ("The MPC will begin actively selling gilts from its portfolio", 0.70),
    ("Quantitative tightening is proceeding in line with the August 2022 framework", 0.65),

    # === RISKS ===
    ("There remain considerable uncertainties around the global economic outlook", 0.35),
    ("Geopolitical tensions represent a significant upside risk to energy prices", 0.35),
    ("The risks to the growth outlook are judged to be weighted to the downside", 0.20),
    ("Risks to inflation from global energy prices have receded", 0.30),
    ("Risks to the inflation outlook are judged to be broadly balanced", 0.40),

    # === BOILERPLATE - NEUTRAL ===
    ("The MPC sets monetary policy to meet the 2 percent inflation target and in a way that helps to sustain growth and employment", 0.50),
    ("The MPC will assess the emerging evidence on the economic outlook and the appropriate path of monetary policy", 0.50),
    ("The MPC will monitor closely the evolution of domestic inflationary pressures", 0.50),
    ("In line with its remit the MPC will continue to take actions to bring inflation back to target", 0.55),
]

print(f"UK training sentences: {len(sentence_training_UK)}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  PAKISTAN — State Bank of Pakistan Monetary Policy Committee (MPC)
#  Communication style: direct references to PKR, SBP, PIBs (Pakistan
#  Investment Bonds), Real Interest Rate (RIR) vs CPI, current account deficit
# ════════════════════════════════════════════════════════════════════════════
sentence_training_PK = [
    # === RATE DECISIONS ===
    ("The Monetary Policy Committee has decided to increase the policy rate by 300 basis points", 0.97),
    ("The MPC has decided to increase the policy rate by 200 basis points", 0.92),
    ("The MPC has decided to increase the policy rate by 100 basis points", 0.85),
    ("The MPC has decided to increase the policy rate by 50 basis points", 0.75),
    ("The MPC decided to maintain the policy rate at its current level", 0.50),
    ("The MPC has decided to reduce the policy rate by 50 basis points", 0.30),
    ("The MPC has decided to reduce the policy rate by 100 basis points", 0.20),
    ("The MPC has decided to reduce the policy rate by 150 basis points", 0.12),
    ("The MPC decided to cut the policy rate significantly to support economic recovery", 0.10),

    # === FORWARD GUIDANCE - HAWKISH ===
    ("The SBP remains committed to achieving its medium-term inflation target of 5 to 7 percent", 0.80),
    ("The MPC noted that real interest rates remain negative and further tightening may be necessary", 0.85),
    ("Given persistent inflationary pressures additional monetary tightening may be required", 0.85),
    ("The MPC will continue to calibrate its policy stance to bring inflation down decisively", 0.80),
    ("The Committee judges that the current real interest rate is still insufficient to anchor inflation expectations", 0.85),
    ("Further rate increases may be needed to ensure that inflation returns to the medium-term target", 0.80),

    # === FORWARD GUIDANCE - DOVISH ===
    ("The MPC expects that the current tightening cycle has largely run its course", 0.25),
    ("The Committee noted significant space for monetary easing given the sharp fall in inflation", 0.20),
    ("As inflation continues to moderate the MPC sees room for further easing", 0.20),
    ("The Committee will assess the evolving economic situation before deciding on any additional adjustments", 0.40),
    ("The MPC will take a cautious and data-driven approach to future rate decisions", 0.40),
    ("The current level of the policy rate is appropriate given the balance of risks to growth and inflation", 0.45),

    # === INFLATION - HAWKISH ===
    ("Headline inflation remains stubbornly high well above the medium-term target", 0.88),
    ("Core inflation has remained sticky and has not declined in line with headline measures", 0.85),
    ("Administered price adjustments have added significantly to inflationary pressure", 0.80),
    ("Energy price increases have fuelled second-round effects across the economy", 0.80),
    ("Inflation expectations are not yet well anchored and remain a key source of risk", 0.90),
    ("Food inflation continues to exert strong upward pressure on the headline CPI", 0.78),
    ("Exchange rate depreciation is feeding through to domestic prices", 0.80),
    ("PKR weakness has raised imported inflation materially", 0.82),

    # === INFLATION - DOVISH ===
    ("Inflation has declined sharply and is now trending towards the medium-term target", 0.18),
    ("CPI inflation fell significantly in the most recent month driven by lower food and energy prices", 0.20),
    ("Inflation is projected to continue its downward trajectory over the next 12 months", 0.22),
    ("Core inflation is also showing early signs of moderation", 0.30),
    ("The pass-through of PKR stabilisation to domestic prices has begun", 0.30),
    ("Inflationary expectations are beginning to moderate in consumer surveys", 0.30),
    ("The impact of energy price adjustments on headline CPI is gradually fading", 0.28),

    # === EXTERNAL SECTOR - HAWKISH ===
    ("The current account deficit has widened beyond sustainable levels", 0.85),
    ("Foreign exchange reserves have declined to critically low levels", 0.88),
    ("Import compression measures may be insufficient to contain external imbalances", 0.80),
    ("Exchange rate pressures remain significant given the large financing needs", 0.82),
    ("The rupee has depreciated sharply and further pressure cannot be ruled out", 0.80),

    # === EXTERNAL SECTOR - DOVISH ===
    ("The current account has moved into surplus reflecting a sharp compression of imports", 0.25),
    ("Foreign exchange reserves have recovered significantly supported by IMF disbursements", 0.20),
    ("The current account deficit has narrowed materially", 0.25),
    ("PKR has stabilised and the exchange rate is broadly consistent with fundamentals", 0.30),
    ("SBP reserves now provide an adequate buffer against external shocks", 0.25),

    # === GROWTH - HAWKISH (demand needs cooling) ===
    ("Domestic demand continues to run ahead of the economy's productive capacity", 0.80),
    ("Private consumption and credit growth remain elevated and are stoking inflationary pressures", 0.80),
    ("GDP growth is expected to remain above potential and requires a tighter policy stance", 0.75),

    # === GROWTH - DOVISH ===
    ("Real GDP growth has slowed considerably and the output gap is now negative", 0.20),
    ("Economic activity has weakened significantly due to floods and external headwinds", 0.15),
    ("Large-scale manufacturing output has contracted", 0.22),
    ("The economy is operating below its potential and supports accommodative monetary policy", 0.18),
    ("Growth prospects have deteriorated and downside risks are elevated", 0.20),

    # === FISCAL / IMF ===
    ("Fiscal consolidation is proceeding in line with the IMF programme commitments", 0.45),
    ("Slippage from the fiscal consolidation path would complicate the inflation outlook", 0.75),
    ("Broad-based structural reforms are essential to place the economy on a sustainable path", 0.45),

    # === BOILERPLATE - NEUTRAL ===
    ("The MPC will continue to closely monitor all developments affecting the outlook for inflation and growth", 0.50),
    ("The SBP remains committed to price stability as the foundation for sustainable growth", 0.55),
    ("The next monetary policy decision will be based on an updated assessment of incoming data", 0.50),
    ("The MPC emphasised that maintaining price stability is critical for macroeconomic stability", 0.55),
]

print(f"PK training sentences: {len(sentence_training_PK)}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  AUSTRALIA — Reserve Bank of Australia (RBA) Board
#  Communication style: plain language, explicit about cash rate target,
#  strong focus on wage growth, trimmed mean inflation, housing market
# ════════════════════════════════════════════════════════════════════════════
sentence_training_AU = [
    # === RATE DECISIONS ===
    ("The Board decided to raise the cash rate target by 50 basis points", 0.92),
    ("The Board decided to raise the cash rate target by 25 basis points", 0.78),
    ("The Board decided to leave the cash rate target unchanged", 0.50),
    ("The Board decided to reduce the cash rate target by 25 basis points", 0.25),
    ("The Board decided to lower the cash rate target to a record low level", 0.08),
    ("The Board voted to increase the cash rate at its fastest pace in decades", 0.92),
    ("The Board considered the case for both increasing and maintaining the cash rate", 0.55),
    ("The Board judged it appropriate to hold the cash rate steady while assessing the impact of prior increases", 0.45),

    # === FORWARD GUIDANCE - HAWKISH ===
    ("The Board expects that further increases in interest rates will be needed over the months ahead", 0.88),
    ("The RBA is resolute in its determination to return inflation to target", 0.82),
    ("Further tightening of monetary policy may be required to ensure that inflation returns to target in a reasonable timeframe", 0.85),
    ("The Board will do what is necessary to bring inflation back to target", 0.85),
    ("The Board has a low tolerance for a slower return of inflation to the 2 to 3 per cent target band", 0.80),
    ("The path of interest rates that is required to bring inflation back to target is uncertain but rates will need to go higher", 0.85),

    # === FORWARD GUIDANCE - DOVISH ===
    ("The Board is prepared to hold rates steady for an extended period if that is what is needed", 0.40),
    ("The Board is not ruling anything in or out with respect to future rate decisions", 0.40),
    ("The Board noted that monetary policy is already in restrictive territory", 0.35),
    ("The Board judged it appropriate to pause and assess the impact of past increases", 0.30),
    ("Future rate decisions will be guided by the data and the evolving assessment of the outlook", 0.40),
    ("There is a credible path back to the inflation target without a recession", 0.30),

    # === INFLATION - HAWKISH ===
    ("Inflation in Australia is too high", 0.82),
    ("Trimmed mean inflation remains well above the 2 to 3 per cent target band", 0.85),
    ("Goods price inflation has been surprisingly persistent", 0.78),
    ("Services inflation is still high and is not expected to decline quickly", 0.82),
    ("Inflation is proving more persistent than expected", 0.82),
    ("Inflation has not yet fallen to the midpoint of the target range despite earlier declines", 0.78),
    ("Domestic cost pressures particularly in services remain significant", 0.78),

    # === INFLATION - DOVISH ===
    ("Inflation has passed its peak and is declining", 0.25),
    ("CPI inflation is expected to continue declining and return to the target band", 0.22),
    ("Trimmed mean inflation is declining in line with the Board's expectations", 0.25),
    ("The decline in goods price inflation is supporting the overall disinflation process", 0.28),
    ("Inflation expectations remain well anchored and consistent with the inflation target", 0.30),
    ("Headline inflation has eased substantially from its peak", 0.20),
    ("Both headline and underlying inflation are now within the target band", 0.15),

    # === LABOR MARKET - HAWKISH ===
    ("The labor market remains tight by historical standards", 0.75),
    ("The unemployment rate is low and labor market conditions remain strong", 0.70),
    ("Wage growth has picked up and is at its highest in many years", 0.78),
    ("The Wage Price Index growth is consistent with above-target inflation if productivity does not improve", 0.82),
    ("Labor market tightness is sustaining strong domestic cost pressures", 0.80),

    # === LABOR MARKET - DOVISH ===
    ("There are early signs that conditions in the labor market are easing", 0.32),
    ("The unemployment rate has risen somewhat from its trough", 0.30),
    ("Job vacancies have declined from their peak", 0.30),
    ("Wage growth is expected to moderate as labor market conditions ease", 0.28),
    ("Forward-looking indicators of labor demand have softened", 0.28),

    # === ECONOMIC ACTIVITY - HAWKISH ===
    ("Household consumption has been more resilient than expected", 0.68),
    ("GDP growth remains above trend", 0.70),
    ("Australian export earnings remain elevated due to high commodity prices", 0.65),
    ("Business investment is strong and is adding to inflationary demand pressure", 0.72),

    # === ECONOMIC ACTIVITY - DOVISH ===
    ("GDP growth has slowed and is now below trend", 0.28),
    ("Household consumption growth has slowed sharply as the cost-of-living squeeze bites", 0.22),
    ("Consumer sentiment is at recessionary levels", 0.18),
    ("Residential construction activity has weakened significantly", 0.25),
    ("The Australian economy has grown more slowly than expected", 0.28),
    ("Real household disposable income has fallen as higher interest rates and inflation weigh", 0.22),

    # === HOUSING MARKET ===
    ("Housing prices are rising sharply and could amplify inflationary pressures", 0.72),
    ("Credit growth remains strong and housing market activity is elevated", 0.70),
    ("Dwelling investment has turned down and housing market conditions have eased", 0.32),
    ("Higher interest rates have led to a significant cooling in housing demand", 0.28),

    # === GLOBAL CONTEXT ===
    ("Global inflationary pressures are proving persistent and central banks are tightening further", 0.78),
    ("The global outlook is subject to significant uncertainty and downside risks", 0.28),
    ("A sharp slowdown in China would weigh materially on Australian exports and growth", 0.22),
    ("Commodity prices have remained elevated supporting Australia's terms of trade", 0.62),

    # === BOILERPLATE - NEUTRAL ===
    ("The Board's priority is to return inflation to the 2 to 3 per cent target band", 0.55),
    ("In doing so the Board will continue to pay close attention to developments in the global economy household spending and wage growth", 0.50),
    ("The Board remains resolute in its determination to return inflation to target and will do what is necessary to achieve that outcome", 0.60),
    ("The cash rate is now at a level that is restrictive", 0.45),
]

print(f"AU training sentences: {len(sentence_training_AU)}")
print(f"\nTotal labeled sentences across all countries: {len(sentence_training_US)+len(sentence_training_UK)+len(sentence_training_PK)+len(sentence_training_AU)}")


## Part 4 — Shared NLP Infrastructure

The `SentDataset` class, training loop wrapper, and statement scorer are identical across countries. Each country gets its own fine-tuned model weights.

In [ ]:
class SentDataset(Dataset):
    """PyTorch dataset for sentence-level hawkish/dovish labels."""
    def __init__(self, texts, scores, tokenizer):
        self.encodings = tokenizer(texts, truncation=True, padding=True,
                                   max_length=128, return_tensors='pt')
        self.labels = torch.tensor(scores, dtype=torch.float32)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)


def train_country_model(sentence_training, country_code, epochs=30):
    """Fine-tune DistilBERT on a country's labeled sentence bank."""
    print(f"\n{'='*60}")
    print(f"Training NLP model for: {country_code}")
    print(f"Training sentences: {len(sentence_training)}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
    model = model.to(device)

    texts  = [s[0] for s in sentence_training]
    scores = [s[1] for s in sentence_training]
    dataset = SentDataset(texts, scores, tokenizer)

    args = TrainingArguments(
        output_dir=f'./model_{country_code}',
        num_train_epochs=epochs,
        per_device_train_batch_size=8,
        learning_rate=2e-5,
        logging_steps=50,
        save_strategy='no',
        use_cpu=True,
    )
    trainer = Trainer(model=model, args=args, train_dataset=dataset)
    trainer.train()
    print(f"{country_code} model training complete")
    return model, tokenizer


def build_skip_patterns(country_code):
    """Return sentence fragments to skip (voting records, names, boilerplate URLs)."""
    common = [
        'media inquiries', 'For release', 'press@', 'Implementation Note',
        'email', 'call 202',
    ]
    country_specific = {
        'US': ['Voting for', 'Voting against', 'Vice Chair', 'Chair;',
               'Brainard', 'Bullard', 'Bowman', 'Harker', 'Mester',
               'Kashkari', 'George', 'Kaplan', 'Clarida', 'Waller',
               'Jefferson', 'Cook', 'Hammack', 'Logan', 'Paulson',
               'Miran', 'Barr', 'Williams', 'Powell'],
        'UK': ['voted for', 'voted against', 'Governor', 'external member',
               'Chief Economist', 'Deputy Governor', 'www.bankofengland',
               'Minutes of the', 'Attending the meeting'],
        'PK': ['Governor SBP', 'Deputy Governor', 'www.sbp.org.pk',
               'Monetary Policy Statement', 'For Immediate Release'],
        'AU': ['Governor', 'Deputy Governor', 'www.rba.gov.au',
               'For Immediate Release', 'voted to raise', 'voted to leave'],
    }
    return common + country_specific.get(country_code, [])


def score_statement(text, model, tokenizer, country_code):
    """Score all content sentences in a central bank statement; return mean hawk score."""
    sentences = text.split('.')
    skip = build_skip_patterns(country_code)
    scores = []

    for sent in sentences:
        sent = sent.strip()
        if len(sent) < 20:
            continue
        if any(p in sent for p in skip):
            continue

        inputs = tokenizer(sent, return_tensors='pt', truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            out = model(**inputs)
        scores.append(out.logits.cpu().numpy()[0][0])

    return float(np.mean(scores)) if scores else 0.5


def run_loo_model(df, target_col='target_60d', features=None):
    """Leave-one-out linear regression. Returns (preds, acts, oos_r2, dir_acc)."""
    if features is None:
        features = ['sent_level_demeaned', 'short_spread']

    md = df[[target_col] + features].dropna()
    X = md[features]
    y = md[target_col]

    loo = LeaveOneOut()
    preds, acts = [], []
    for tr, te in loo.split(X):
        reg = LinearRegression()
        reg.fit(X.iloc[tr], y.iloc[tr])
        preds.append(reg.predict(X.iloc[te])[0])
        acts.append(y.iloc[te].values[0])

    preds, acts = np.array(preds), np.array(acts)
    ss_res = np.sum((acts - preds) ** 2)
    ss_tot = np.sum((acts - acts.mean()) ** 2)
    oos_r2 = 1 - ss_res / ss_tot
    dir_acc = np.mean(np.sign(preds) == np.sign(acts))
    return preds, acts, oos_r2, dir_acc, md

print("Shared NLP infrastructure defined.")


## Part 5 — Country Pipelines

### 5a. United States — FOMC Data & Model

In [ ]:
# ── US: Pull FRED macro data ──────────────────────────────────────────────
gdp_us     = fred.get_series('A191RL1Q225SBEA')
cpi_us     = fred.get_series('CPIAUCSL')
core_pce   = fred.get_series('PCEPILFE')
unrate_us  = fred.get_series('UNRATE')
ff_rate    = fred.get_series('DFF')
two_year   = fred.get_series('DGS2')
one_year   = fred.get_series('DGS1')

core_pce_yoy = core_pce.pct_change(12) * 100
twoy_ff_spread = two_year - ff_rate

us_macro = pd.DataFrame({
    'gdp': gdp_us, 'cpi': cpi_us, 'core_pce_yoy': core_pce_yoy,
    'unemployment': unrate_us, 'policy_rate': ff_rate,
    '2y_yield': two_year, '1y_yield': one_year, 'short_spread': twoy_ff_spread,
}).resample('ME').last().ffill()

print("US macro data shape:", us_macro.shape)


In [ ]:
# ── US: Load statements from disk (scrape only if not yet saved) ──────────
statements_us_raw = load_statements('us')

if not statements_us_raw:
    print("No saved US statements found — running scraper (this may take several minutes)...")
    statements_us_raw = scrape_statements('us')
else:
    print(f"Loaded {len(statements_us_raw)} US statements from disk.")
    labels = get_labels('us')
    print(f"Date range: {labels[0]} to {labels[-1]}")


In [ ]:
# ── US: Pull FRED macro data ──────────────────────────────────────────────
gdp_us     = fred.get_series('A191RL1Q225SBEA')
cpi_us     = fred.get_series('CPIAUCSL')
core_pce   = fred.get_series('PCEPILFE')
unrate_us  = fred.get_series('UNRATE')
ff_rate    = fred.get_series('DFF')
two_year   = fred.get_series('DGS2')
one_year   = fred.get_series('DGS1')

core_pce_yoy   = core_pce.pct_change(12) * 100
twoy_ff_spread = two_year - ff_rate

us_macro = pd.DataFrame({
    'gdp': gdp_us, 'cpi': cpi_us, 'core_pce_yoy': core_pce_yoy,
    'unemployment': unrate_us, 'policy_rate': ff_rate,
    '2y_yield': two_year, '1y_yield': one_year, 'short_spread': twoy_ff_spread,
}).resample('ME').last().ffill()

# Build FOMC dataframe from loaded statements
us_rows = []
for date_str, text in statements_us_raw.items():
    date = pd.to_datetime(date_str)
    try:
        macro_date = us_macro.index[us_macro.index <= date][-1]
        row = us_macro.loc[macro_date]
        us_rows.append({
            'date': date, 'statement': text,
            'policy_rate': row['policy_rate'],
            '1y_yield': row['1y_yield'],
            'short_spread': row['short_spread'],
        })
    except Exception:
        pass

fomc_df_us = pd.DataFrame(us_rows).sort_values('date').set_index('date')

# 60-day yield change target
one_year_daily = fred.get_series('DGS1')
fomc_df_us['target_60d'] = None
for date in fomc_df_us.index:
    try:
        loc = one_year_daily.index.get_indexer([date], method='ffill')[0]
        if loc + 44 < len(one_year_daily):
            fomc_df_us.loc[date, 'target_60d'] = (
                one_year_daily.iloc[loc + 44] - one_year_daily.iloc[loc])
    except Exception:
        pass
fomc_df_us['target_60d'] = fomc_df_us['target_60d'].astype(float)

print(f"US dataframe: {fomc_df_us.shape}, target obs: {fomc_df_us['target_60d'].notna().sum()}")


In [ ]:
# ── US: Train NLP model & score statements ────────────────────────────────
model_us, tok_us = train_country_model(sentence_training_US, 'US', epochs=30)

print("Scoring US statements...")
for date, row in fomc_df_us.iterrows():
    if pd.notna(row.get('statement')):
        fomc_df_us.loc[date, 'sent_level'] = score_statement(
            row['statement'], model_us, tok_us, 'US')

fomc_df_us['sent_level'] = fomc_df_us['sent_level'].astype(float)
fomc_df_us['sent_level_demeaned'] = fomc_df_us['sent_level'] - fomc_df_us['sent_level'].mean()
print("US scoring complete")

preds_us, acts_us, oos_r2_us, dir_us, md_us = run_loo_model(fomc_df_us)
print(f"US — OOS R²: {oos_r2_us:.3f}  |  Directional accuracy: {dir_us:.1%}")


### 5b. United Kingdom — MPC Data & Model

In [ ]:
# ── UK: Pull FRED macro data ──────────────────────────────────────────────
# Note: FRED series for BoE Bank Rate is monthly; 1Y gilt proxy is also monthly
boe_rate   = fred.get_series('BOERUKM')
uk_cpi     = fred.get_series('GBRCPIALLMINMEI')
uk_unrate  = fred.get_series('LRHUTTTTGBM156S')

# 1-year gilt proxy: use IRLTLT01GBM156N (long-term rate monthly, closest available)
# For short-end we use the 3-month T-bill IRSTCB01GBM156N as the spread base
try:
    uk_1y = fred.get_series('IRLTLT01GBM156N')
except Exception:
    uk_1y = boe_rate  # fallback

try:
    uk_3m = fred.get_series('IRSTCB01GBM156N')
except Exception:
    uk_3m = boe_rate

uk_cpi_yoy = uk_cpi.pct_change(12) * 100
uk_short_spread = uk_3m - boe_rate   # 3M bill vs Bank Rate (market pricing of near moves)

uk_macro = pd.DataFrame({
    'policy_rate': boe_rate,
    'cpi_yoy': uk_cpi_yoy,
    'unemployment': uk_unrate,
    '1y_yield': uk_1y,
    'short_spread': uk_short_spread,
}).resample('ME').last().ffill()

print("UK macro data shape:", uk_macro.shape)


In [ ]:
# ── UK: Load statements from disk (scrape only if not yet saved) ──────────
statements_uk_raw = load_statements('uk')

if not statements_uk_raw:
    print("No saved UK statements found — running scraper (this may take several minutes)...")
    statements_uk_raw = scrape_statements('uk')
else:
    print(f"Loaded {len(statements_uk_raw)} UK statements from disk.")
    labels = get_labels('uk')
    print(f"Date range: {labels[0]} to {labels[-1]}")


In [ ]:
# ── UK: Pull FRED macro data ──────────────────────────────────────────────
boe_rate   = fred.get_series('BOERUKM')
uk_cpi     = fred.get_series('GBRCPIALLMINMEI')
uk_unrate  = fred.get_series('LRHUTTTTGBM156S')

try:
    uk_1y = fred.get_series('IRLTLT01GBM156N')
except Exception:
    uk_1y = boe_rate

try:
    uk_3m = fred.get_series('IRSTCB01GBM156N')
except Exception:
    uk_3m = boe_rate

uk_cpi_yoy      = uk_cpi.pct_change(12) * 100
uk_short_spread = uk_3m - boe_rate

uk_macro = pd.DataFrame({
    'policy_rate': boe_rate, 'cpi_yoy': uk_cpi_yoy,
    'unemployment': uk_unrate, '1y_yield': uk_1y, 'short_spread': uk_short_spread,
}).resample('ME').last().ffill()

# Build MPC dataframe from loaded statements
uk_rows = []
for date_str, text in statements_uk_raw.items():
    date = pd.to_datetime(date_str)
    try:
        macro_date = uk_macro.index[uk_macro.index <= date][-1]
        row = uk_macro.loc[macro_date]
        uk_rows.append({
            'date': date, 'statement': text,
            'policy_rate': row['policy_rate'],
            '1y_yield': row['1y_yield'],
            'short_spread': row['short_spread'],
        })
    except Exception:
        pass

fomc_df_uk = pd.DataFrame(uk_rows).sort_values('date').set_index('date') if uk_rows else pd.DataFrame()

if len(fomc_df_uk) > 0:
    uk_yield_series = uk_macro['1y_yield'].dropna()
    fomc_df_uk['target_60d'] = None
    for date in fomc_df_uk.index:
        try:
            future_date = date + pd.DateOffset(days=60)
            fi = uk_yield_series.index.get_indexer([future_date], method='nearest')[0]
            ci = uk_yield_series.index.get_indexer([date], method='nearest')[0]
            fomc_df_uk.loc[date, 'target_60d'] = (
                uk_yield_series.iloc[fi] - uk_yield_series.iloc[ci])
        except Exception:
            pass
    fomc_df_uk['target_60d'] = fomc_df_uk['target_60d'].astype(float)

print(f"UK dataframe: {fomc_df_uk.shape}, target obs: {fomc_df_uk['target_60d'].notna().sum() if len(fomc_df_uk) > 0 else 0}")


In [ ]:
# ── UK: Train NLP model & score statements ────────────────────────────────
model_uk, tok_uk = train_country_model(sentence_training_UK, 'UK', epochs=30)

print("Scoring UK statements...")
for date, row in fomc_df_uk.iterrows():
    if pd.notna(row.get('statement')):
        fomc_df_uk.loc[date, 'sent_level'] = score_statement(
            row['statement'], model_uk, tok_uk, 'UK')

if len(fomc_df_uk) > 0:
    fomc_df_uk['sent_level'] = fomc_df_uk['sent_level'].astype(float)
    fomc_df_uk['sent_level_demeaned'] = fomc_df_uk['sent_level'] - fomc_df_uk['sent_level'].mean()
    preds_uk, acts_uk, oos_r2_uk, dir_uk, md_uk = run_loo_model(fomc_df_uk)
    print(f"UK — OOS R²: {oos_r2_uk:.3f}  |  Directional accuracy: {dir_uk:.1%}")
else:
    print("No UK data collected — check scraper / network access")
    oos_r2_uk, dir_uk = None, None


### 5c. Pakistan — SBP MPC Data & Model

In [ ]:
# ── Pakistan: Pull FRED macro data ───────────────────────────────────────
try:
    sbp_rate = fred.get_series('IRSTCB01PKM156N')
except Exception:
    print("SBP rate not on FRED; using placeholder zeros")
    sbp_rate = pd.Series(dtype=float)

try:
    pk_1y = fred.get_series('IRLTLT01PKM156N')
except Exception:
    pk_1y = sbp_rate

try:
    pk_cpi = fred.get_series('PAKCP1020M086NEST')
    pk_cpi_yoy = pk_cpi.pct_change(12) * 100
except Exception:
    pk_cpi_yoy = pd.Series(dtype=float)

try:
    pk_unrate = fred.get_series('LRUN64TTPKM156S')
except Exception:
    pk_unrate = pd.Series(dtype=float)

pk_short_spread = (pk_1y - sbp_rate) if (len(pk_1y) > 0 and len(sbp_rate) > 0) else pd.Series(dtype=float)

pk_macro = pd.DataFrame({
    'policy_rate': sbp_rate,
    'cpi_yoy': pk_cpi_yoy,
    'unemployment': pk_unrate,
    '1y_yield': pk_1y,
    'short_spread': pk_short_spread,
}).resample('ME').last().ffill()

print("Pakistan macro data shape:", pk_macro.shape)


In [ ]:
# ── Pakistan: Load statements from disk (scrape only if not yet saved) ────
statements_pk_raw = load_statements('pk')

if not statements_pk_raw:
    print("No saved Pakistan statements found — running scraper...")
    print("Note: Pakistan scraper requires pdfplumber (pip install pdfplumber)")
    statements_pk_raw = scrape_statements('pk')
else:
    print(f"Loaded {len(statements_pk_raw)} Pakistan statements from disk.")
    labels = get_labels('pk')
    print(f"Date range: {labels[0]} to {labels[-1]}")


In [ ]:
# ── Pakistan: Pull FRED macro data ───────────────────────────────────────
try:
    sbp_rate = fred.get_series('IRSTCB01PKM156N')
except Exception:
    sbp_rate = pd.Series(dtype=float)

try:
    pk_1y = fred.get_series('IRLTLT01PKM156N')
except Exception:
    pk_1y = sbp_rate

try:
    pk_cpi     = fred.get_series('PAKCP1020M086NEST')
    pk_cpi_yoy = pk_cpi.pct_change(12) * 100
except Exception:
    pk_cpi_yoy = pd.Series(dtype=float)

try:
    pk_unrate = fred.get_series('LRUN64TTPKM156S')
except Exception:
    pk_unrate = pd.Series(dtype=float)

pk_short_spread = (pk_1y - sbp_rate) if (len(pk_1y) > 0 and len(sbp_rate) > 0) else pd.Series(dtype=float)

pk_macro = pd.DataFrame({
    'policy_rate': sbp_rate, 'cpi_yoy': pk_cpi_yoy,
    'unemployment': pk_unrate, '1y_yield': pk_1y, 'short_spread': pk_short_spread,
}).resample('ME').last().ffill()

# Build MPC dataframe from loaded statements
pk_rows = []
for date_str, text in statements_pk_raw.items():
    date = pd.to_datetime(date_str)
    try:
        macro_date = pk_macro.index[pk_macro.index <= date][-1]
        row = pk_macro.loc[macro_date]
        pk_rows.append({
            'date': date, 'statement': text,
            'policy_rate': row.get('policy_rate', float('nan')),
            '1y_yield': row.get('1y_yield', float('nan')),
            'short_spread': row.get('short_spread', float('nan')),
        })
    except Exception:
        pass

fomc_df_pk = pd.DataFrame(pk_rows).sort_values('date').set_index('date') if pk_rows else pd.DataFrame()

if len(fomc_df_pk) > 0:
    pk_yield_series = pk_macro['1y_yield'].dropna()
    fomc_df_pk['target_60d'] = None
    for date in fomc_df_pk.index:
        try:
            future_date = date + pd.DateOffset(days=60)
            fi = pk_yield_series.index.get_indexer([future_date], method='nearest')[0]
            ci = pk_yield_series.index.get_indexer([date], method='nearest')[0]
            fomc_df_pk.loc[date, 'target_60d'] = pk_yield_series.iloc[fi] - pk_yield_series.iloc[ci]
        except Exception:
            pass
    fomc_df_pk['target_60d'] = fomc_df_pk['target_60d'].astype(float)

print(f"Pakistan dataframe: {fomc_df_pk.shape}, target obs: {fomc_df_pk['target_60d'].notna().sum() if len(fomc_df_pk) > 0 else 0}")


In [ ]:
# ── Pakistan: Train NLP model & score statements ──────────────────────────
model_pk, tok_pk = train_country_model(sentence_training_PK, 'PK', epochs=30)

if len(fomc_df_pk) > 0:
    print("Scoring Pakistan statements...")
    for date, row in fomc_df_pk.iterrows():
        if pd.notna(row.get('statement')):
            fomc_df_pk.loc[date, 'sent_level'] = score_statement(
                row['statement'], model_pk, tok_pk, 'PK')

    fomc_df_pk['sent_level'] = fomc_df_pk['sent_level'].astype(float)
    fomc_df_pk['sent_level_demeaned'] = fomc_df_pk['sent_level'] - fomc_df_pk['sent_level'].mean()

    if fomc_df_pk[['target_60d','sent_level_demeaned','short_spread']].dropna().__len__() >= 5:
        preds_pk, acts_pk, oos_r2_pk, dir_pk, md_pk = run_loo_model(fomc_df_pk)
        print(f"PK — OOS R²: {oos_r2_pk:.3f}  |  Directional accuracy: {dir_pk:.1%}")
    else:
        print("Not enough Pakistan observations for LOO — run after collecting more statements")
        oos_r2_pk, dir_pk = None, None
else:
    print("No Pakistan data collected — check PDF scraper / pdfminer installation")
    oos_r2_pk, dir_pk = None, None


### 5d. Australia — RBA Board Data & Model

In [ ]:
# ── Australia: Pull FRED macro data ──────────────────────────────────────
try:
    rba_rate = fred.get_series('RBXMNYCBD')
except Exception:
    try:
        rba_rate = fred.get_series('IR3TIB01AUM156N')
    except Exception:
        print("RBA rate not found on FRED")
        rba_rate = pd.Series(dtype=float)

try:
    au_1y = fred.get_series('IR3TIB01AUM156N')   # 3M interbank as 1Y proxy
except Exception:
    au_1y = rba_rate

try:
    au_cpi = fred.get_series('AUSCPIALLMINMEI')
    au_cpi_yoy = au_cpi.pct_change(12) * 100
except Exception:
    au_cpi_yoy = pd.Series(dtype=float)

try:
    au_unrate = fred.get_series('LRUN64TTAUM156S')
except Exception:
    au_unrate = pd.Series(dtype=float)

au_short_spread = (au_1y - rba_rate) if (len(au_1y) > 0 and len(rba_rate) > 0) else pd.Series(dtype=float)

au_macro = pd.DataFrame({
    'policy_rate': rba_rate,
    'cpi_yoy': au_cpi_yoy,
    'unemployment': au_unrate,
    '1y_yield': au_1y,
    'short_spread': au_short_spread,
}).resample('ME').last().ffill()

print("Australia macro data shape:", au_macro.shape)


In [ ]:
# ── Australia: Load statements from disk (scrape only if not yet saved) ───
statements_au_raw = load_statements('au')

if not statements_au_raw:
    print("No saved Australia statements found — running scraper...")
    statements_au_raw = scrape_statements('au')
else:
    print(f"Loaded {len(statements_au_raw)} Australia statements from disk.")
    labels = get_labels('au')
    print(f"Date range: {labels[0]} to {labels[-1]}")


In [ ]:
# ── Australia: Pull FRED macro data ──────────────────────────────────────
try:
    rba_rate = fred.get_series('RBXMNYCBD')
except Exception:
    try:
        rba_rate = fred.get_series('IR3TIB01AUM156N')
    except Exception:
        rba_rate = pd.Series(dtype=float)

try:
    au_1y = fred.get_series('IR3TIB01AUM156N')
except Exception:
    au_1y = rba_rate

try:
    au_cpi     = fred.get_series('AUSCPIALLMINMEI')
    au_cpi_yoy = au_cpi.pct_change(12) * 100
except Exception:
    au_cpi_yoy = pd.Series(dtype=float)

try:
    au_unrate = fred.get_series('LRUN64TTAUM156S')
except Exception:
    au_unrate = pd.Series(dtype=float)

au_short_spread = (au_1y - rba_rate) if (len(au_1y) > 0 and len(rba_rate) > 0) else pd.Series(dtype=float)

au_macro = pd.DataFrame({
    'policy_rate': rba_rate, 'cpi_yoy': au_cpi_yoy,
    'unemployment': au_unrate, '1y_yield': au_1y, 'short_spread': au_short_spread,
}).resample('ME').last().ffill()

# Build RBA dataframe from loaded statements
au_rows = []
for date_str, text in statements_au_raw.items():
    date = pd.to_datetime(date_str)
    try:
        macro_date = au_macro.index[au_macro.index <= date][-1]
        row = au_macro.loc[macro_date]
        au_rows.append({
            'date': date, 'statement': text,
            'policy_rate': row.get('policy_rate', float('nan')),
            '1y_yield': row.get('1y_yield', float('nan')),
            'short_spread': row.get('short_spread', float('nan')),
        })
    except Exception:
        pass

fomc_df_au = pd.DataFrame(au_rows).sort_values('date').set_index('date') if au_rows else pd.DataFrame()

if len(fomc_df_au) > 0:
    au_yield_series = au_macro['1y_yield'].dropna()
    fomc_df_au['target_60d'] = None
    for date in fomc_df_au.index:
        try:
            future_date = date + pd.DateOffset(days=60)
            fi = au_yield_series.index.get_indexer([future_date], method='nearest')[0]
            ci = au_yield_series.index.get_indexer([date], method='nearest')[0]
            fomc_df_au.loc[date, 'target_60d'] = au_yield_series.iloc[fi] - au_yield_series.iloc[ci]
        except Exception:
            pass
    fomc_df_au['target_60d'] = fomc_df_au['target_60d'].astype(float)

print(f"Australia dataframe: {fomc_df_au.shape}, target obs: {fomc_df_au['target_60d'].notna().sum() if len(fomc_df_au) > 0 else 0}")


In [ ]:
# ── Australia: Train NLP model & score statements ─────────────────────────
model_au, tok_au = train_country_model(sentence_training_AU, 'AU', epochs=30)

if len(fomc_df_au) > 0:
    print("Scoring Australia statements...")
    for date, row in fomc_df_au.iterrows():
        if pd.notna(row.get('statement')):
            fomc_df_au.loc[date, 'sent_level'] = score_statement(
                row['statement'], model_au, tok_au, 'AU')

    fomc_df_au['sent_level'] = fomc_df_au['sent_level'].astype(float)
    fomc_df_au['sent_level_demeaned'] = fomc_df_au['sent_level'] - fomc_df_au['sent_level'].mean()

    if fomc_df_au[['target_60d','sent_level_demeaned','short_spread']].dropna().__len__() >= 5:
        preds_au, acts_au, oos_r2_au, dir_au, md_au = run_loo_model(fomc_df_au)
        print(f"AU — OOS R²: {oos_r2_au:.3f}  |  Directional accuracy: {dir_au:.1%}")
    else:
        print("Not enough Australia observations for LOO")
        oos_r2_au, dir_au = None, None
else:
    print("No Australia data collected — check RBA scraper")
    oos_r2_au, dir_au = None, None


## Part 6 — Test Statements

Synthetic test statements for each country. These are written to be stylistically faithful to each central bank's real communications and span a range of hawkish/dovish stances for model interrogation.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  TEST STATEMENTS — one hawkish, one neutral, one dovish per country
#  Stylistically faithful to real MPC/Board communications
# ════════════════════════════════════════════════════════════════════════════

test_statements = {

    # ── UK ───────────────────────────────────────────────────────────────
    'UK_hawkish': """
        The Monetary Policy Committee voted by a majority of 7-2 to increase Bank Rate by 25 basis points,
        to 5.25 percent. Two members preferred to maintain Bank Rate at 5 percent.
        Twelve-month CPI inflation was 8.7 percent in April 2023, well above the MPC's 2 percent target.
        Services inflation remained high at 6.9 percent and wage growth in the private sector was around 7 percent,
        materially above levels consistent with the inflation target on a sustained basis.
        The MPC will take the actions necessary to return inflation to the 2 percent target sustainably in the medium term.
        Further tightening in monetary policy may be required for a sustainable return of inflation to target,
        and the Committee will be particularly alert to signs of more persistent inflationary pressures.
        The labor market remains tight by historical standards, and domestic inflationary pressures have proved more persistent than expected.
        Given the significant increase in Bank Rate since late 2021 the Committee will continue to monitor
        closely the impact of the tightening cycle on credit conditions and activity.
    """,

    'UK_neutral': """
        The Monetary Policy Committee voted by a majority of 5-4 to maintain Bank Rate at 5.25 percent.
        Four members preferred to reduce Bank Rate by 25 basis points to 5 percent.
        CPI inflation fell to 3.2 percent in March 2024, down from its peak of over 11 percent in late 2022.
        The MPC judged that monetary policy is likely to need to be restrictive for an extended period of time.
        Services price inflation remained elevated at 6 percent. The labor market has eased somewhat but remains relatively tight.
        The Committee noted that the risks to the inflation outlook are now judged to be broadly balanced.
        Future decisions will be sensitive to the incoming economic data and the evolving assessment of the outlook.
        The MPC will adjust monetary policy as warranted by the evidence.
    """,

    'UK_dovish': """
        The Monetary Policy Committee voted by a majority of 6-3 to reduce Bank Rate by 25 basis points,
        to 4.75 percent. Three members preferred to maintain Bank Rate at 5 percent.
        Headline CPI inflation fell to 1.7 percent in September 2024, below the MPC's 2 percent target
        for the first time in over three years.
        The Committee noted that monetary policy is already in restrictive territory and that there are
        increasing signs that the restrictive stance is working to return inflation sustainably to target.
        Services inflation has eased somewhat, although it remains somewhat above levels consistent with the target.
        The labor market has loosened and indicators of pay growth have moderated, although they remain elevated.
        GDP growth has been below potential and the output gap is slightly negative.
        A gradual and limited approach to further reducing monetary policy stimulus remains appropriate.
        The MPC is prepared to adjust the stance of monetary policy as appropriate if the outlook changes.
    """,

    # ── PAKISTAN ─────────────────────────────────────────────────────────
    'PK_hawkish': """
        The Monetary Policy Committee met on 14th October 2022 and decided to increase the policy rate
        by 100 basis points to 16 percent. This decision reflects the Committee's assessment that
        headline inflation remains stubbornly high well above the medium-term target of 5 to 7 percent.
        CPI inflation surged to 23.2 percent in September 2022, driven by administered energy price increases,
        exchange rate depreciation, and elevated global commodity prices feeding through to domestic prices.
        Core inflation has remained sticky and has not declined in line with headline measures.
        The MPC noted that real interest rates remain significantly negative and further tightening is necessary
        to anchor inflation expectations and achieve a sustained reduction in inflation.
        The current account deficit has widened beyond sustainable levels and foreign exchange reserves
        have declined to critically low levels, underscoring the need for continued monetary tightening.
        PKR weakness has raised imported inflation materially. The Committee remains strongly committed to
        achieving price stability and will continue to calibrate its policy stance accordingly.
    """,

    'PK_neutral': """
        The Monetary Policy Committee met on 30th January 2023 and decided to maintain the policy rate
        at 17 percent. The Committee judges that the current level of the policy rate is appropriate
        given the balance of risks to growth and inflation.
        CPI inflation was 27.6 percent in December 2022, reflecting the combined effect of energy price adjustments,
        exchange rate pass-through, and food supply disruptions caused by the devastating 2022 floods.
        Core inflation also rose but is showing some early signs of moderation.
        Real GDP growth has slowed considerably and the output gap is now negative.
        The current account deficit has narrowed materially due to import compression measures.
        The Committee will take a cautious and data-driven approach to future rate decisions and will assess
        the evolving economic situation before deciding on any additional adjustments.
        Fiscal consolidation in line with the IMF programme commitments is critical to contain aggregate demand.
    """,

    'PK_dovish': """
        The Monetary Policy Committee met on 27th July 2024 and decided to reduce the policy rate
        by 100 basis points to 19.5 percent. The Committee noted significant space for monetary easing
        given the sharp fall in inflation and improving external accounts.
        CPI inflation has declined sharply to 12.6 percent in June 2024 from a peak of nearly 38 percent
        in May 2023, and is now trending towards the medium-term target.
        Core inflation is also showing early signs of moderation. SBP reserves have recovered significantly
        supported by IMF disbursements and the current account has moved into surplus.
        PKR has stabilised and the exchange rate is broadly consistent with fundamentals.
        As inflation continues to moderate the MPC sees room for further easing in a calibrated manner.
        Real GDP growth has weakened and the economy is operating below its potential.
        The SBP remains committed to price stability as the foundation for sustainable growth.
    """,

    # ── AUSTRALIA ────────────────────────────────────────────────────────
    'AU_hawkish': """
        At its meeting today, the Reserve Bank of Australia Board decided to increase the cash rate target
        by 50 basis points to 1.85 per cent and the interest rate on Exchange Settlement balances by 50 basis points.
        Inflation in Australia is too high. Global factors explain much of the increase in inflation,
        but domestic factors are also playing a role. Trimmed mean inflation was 4.9 per cent over the year to the June quarter.
        The labor market is tight and many firms are having difficulty hiring workers. The unemployment rate
        was 3.5 per cent in June, the lowest rate in nearly 50 years. Wage growth has picked up and
        the Wage Price Index is at its highest growth rate since 2012.
        The Board expects that further increases in interest rates will be needed over the months ahead,
        but it is not on a pre-set path. The RBA is resolute in its determination to return inflation to target.
        A further increase in inflation would risk a more damaging and disruptive correction later.
    """,

    'AU_neutral': """
        At its meeting today, the Reserve Bank of Australia Board decided to leave the cash rate target unchanged
        at 4.35 per cent. The Board remains resolute in its determination to return inflation to target
        and will do what is necessary to achieve that outcome.
        Inflation is declining but is still some way above the midpoint of the 2-3 per cent target range.
        Trimmed mean inflation was 3.5 per cent over the year to the December quarter, down from its peak.
        The labor market remains tight by historical standards, though there are early signs that conditions
        in the labor market are easing. Forward-looking indicators of labor demand have softened.
        Household consumption growth has slowed as higher interest rates and cost of living pressures bite.
        The path of interest rates needed to return inflation to target remains uncertain.
        The Board will continue to closely monitor developments in the global economy and domestic conditions.
        Further tightening of monetary policy may be required to ensure inflation returns to target in a reasonable timeframe.
    """,

    'AU_dovish': """
        At its meeting today, the Reserve Bank of Australia Board decided to reduce the cash rate target
        by 25 basis points to 4.10 per cent and the interest rate on Exchange Settlement balances by 25 basis points.
        The Board judges that this reduction is appropriate given the continued decline in inflation
        and the softening in the labor market and economic activity.
        Headline CPI inflation has eased substantially from its peak and both headline and underlying inflation
        are now within the target band. The Board judged that monetary policy is already in restrictive territory
        and that the case for a further reduction in the cash rate was compelling.
        GDP growth has slowed and is now below trend. Consumer confidence has fallen sharply
        and real household disposable income has declined. There is a credible path back to the inflation target
        without a recession but it requires supporting activity at the margin.
        The Board is prepared to hold rates steady for an extended period if that is what is needed,
        but today judged that the balance of risks had shifted.
    """,
}

print(f"Test statements defined: {list(test_statements.keys())}")


In [ ]:
# ── Score test statements with the appropriate country model ──────────────

def score_test_statement(text, country_code):
    model_map = {'UK': (model_uk, tok_uk), 'PK': (model_pk, tok_pk), 'AU': (model_au, tok_au)}
    if country_code not in model_map:
        return None
    model, tok = model_map[country_code]
    return score_statement(text, model, tok, country_code)

results_test = []
for key, text in test_statements.items():
    country = key.split('_')[0]
    stance   = key.split('_')[1]
    score = score_test_statement(text, country)
    results_test.append({
        'Key': key,
        'Country': country,
        'Stance': stance,
        'Hawk Score (0=dove, 1=hawk)': round(score, 4) if score is not None else 'N/A',
        'Label': '🟥 Hawkish' if score and score > 0.6 else ('🟩 Dovish' if score and score < 0.4 else '🟨 Neutral') if score else '-'
    })

results_df = pd.DataFrame(results_test)
print("\nTest Statement Scores:")
print(results_df.to_string(index=False))


## Part 7 — Cross-Country Performance Dashboard

In [ ]:
# ── Summary comparison table ──────────────────────────────────────────────
summary = []
country_data = [
    ('US', oos_r2_us, dir_us, len(fomc_df_us)),
    ('UK', oos_r2_uk if 'oos_r2_uk' in dir() else None,
           dir_uk if 'dir_uk' in dir() else None,
           len(fomc_df_uk) if len(fomc_df_uk) > 0 else 0),
    ('PK', oos_r2_pk if 'oos_r2_pk' in dir() else None,
           dir_pk if 'dir_pk' in dir() else None,
           len(fomc_df_pk) if len(fomc_df_pk) > 0 else 0),
    ('AU', oos_r2_au if 'oos_r2_au' in dir() else None,
           dir_au if 'dir_au' in dir() else None,
           len(fomc_df_au) if len(fomc_df_au) > 0 else 0),
]

print(f"{'Country':<8} {'OOS R²':>8} {'Dir %':>8} {'Obs':>6}")
print('-' * 35)
for cc, r2, da, n in country_data:
    r2_str = f"{r2:.3f}" if r2 is not None else "  N/A "
    da_str = f"{da:.1%}" if da is not None else "  N/A"
    print(f"{cc:<8} {r2_str:>8} {da_str:>8} {n:>6}")


In [ ]:
# ── Visualization: sentiment time series + test statement scores ──────────
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
fig.suptitle('FedSignal — Multi-Country Central Bank NLP Dashboard', fontsize=15, fontweight='bold')

country_plot_data = [
    ('US', fomc_df_us, preds_us, acts_us, '#3498db'),
    ('UK', fomc_df_uk if len(fomc_df_uk) > 0 else None, None, None, '#e74c3c'),
    ('PK', fomc_df_pk if len(fomc_df_pk) > 0 else None, None, None, '#2ecc71'),
    ('AU', fomc_df_au if len(fomc_df_au) > 0 else None, None, None, '#f39c12'),
]

for ax, (cc, df, preds, acts, color) in zip(axes.flatten(), country_plot_data):
    ax.set_title(f"{cc} — Sentiment (demeaned) over time", fontsize=11)
    if df is not None and len(df) > 0 and 'sent_level_demeaned' in df.columns:
        valid = df['sent_level_demeaned'].dropna()
        ax.bar(valid.index, valid.values, width=25, color=color, alpha=0.75, edgecolor='black', linewidth=0.3)
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
        if preds is not None and acts is not None:
            ax2 = ax.twinx()
            ax2.plot(df[['target_60d','sent_level_demeaned']].dropna().index, preds,
                     color='black', linewidth=1.2, label='Predicted', alpha=0.7)
            ax2.set_ylabel('60d yield change (%)', fontsize=8)
    else:
        ax.text(0.5, 0.5, 'No data\n(check scraper)', ha='center', va='center',
                transform=ax.transAxes, fontsize=12, color='gray')
    ax.set_ylabel('Sentiment score (demeaned)')
    ax.set_xlabel('Date')

plt.tight_layout()
plt.show()


In [ ]:
# ── Visualization: Test statement hawk scores ─────────────────────────────
if results_test:
    fig2, ax = plt.subplots(figsize=(12, 5))
    labels = [r['Key'] for r in results_test]
    scores_plot = [r['Hawk Score (0=dove, 1=hawk)'] if isinstance(r['Hawk Score (0=dove, 1=hawk)'], float) else 0.5
                   for r in results_test]
    bar_colors = ['#e74c3c' if s > 0.6 else '#2ecc71' if s < 0.4 else '#f39c12' for s in scores_plot]
    bars = ax.bar(labels, scores_plot, color=bar_colors, edgecolor='black', linewidth=0.5)
    ax.axhline(0.5, color='black', linestyle='--', linewidth=0.8, label='Neutral (0.5)')
    ax.axhline(0.6, color='red',   linestyle=':', linewidth=0.8, alpha=0.6, label='Hawkish threshold')
    ax.axhline(0.4, color='green', linestyle=':', linewidth=0.8, alpha=0.6, label='Dovish threshold')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Hawkishness Score (0 = Dovish, 1 = Hawkish)')
    ax.set_title('Test Statement Scores by Country Model')
    ax.legend(fontsize=9)
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

print("\nDone. Each country's DistilBERT model was trained on its own labeled sentence bank.")
print("To predict a new statement, call score_statement(text, model_XX, tok_XX, 'XX')")
print("with the appropriate country model and tokenizer.")
